# Main Analysis: Classification Algorithms Comparison

This notebook implements, trains, and compares k-NN, Random Forest, and Naive Bayes classifiers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
import os

# Import our custom implementations
from knn_classifier import KNNClassifier
from random_forest import RandomForest
from naive_bayes import NaiveBayes
from evaluation_utils import (
    calculate_metrics, plot_confusion_matrix, plot_roc_curve,
    plot_learning_curve, plot_feature_importance, print_metrics
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## Load Preprocessed Data

Load the data that was preprocessed in the EDA notebook.

In [ ]:
# Load preprocessed data
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train_scaled']
X_val = data['X_val_scaled']
X_test = data['X_test_scaled']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']
feature_names = data['feature_names']

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeature names: {feature_names}")
print(f"\nClass distribution in training: {np.bincount(y_train)}")
print(f"Class distribution in validation: {np.bincount(y_val)}")
print(f"Class distribution in test: {np.bincount(y_test)}")

## 1. k-NN Classifier

### Hyperparameter Tuning

We'll tune k-NN with different k values (1, 3, 5, 7, 10), distance metrics (Euclidean, Manhattan), and voting schemes (weighted vs majority).

In [ ]:
# Tune k-NN
print("="*60)
print("Tuning k-NN Classifier")
print("="*60)

k_values = [1, 3, 5, 7, 10]
knn_results = []

for metric in ['euclidean', 'manhattan']:
    for k in k_values:
        for weighted in [True, False]:
            knn = KNNClassifier(k=k, distance_metric=metric, weighted=weighted)
            knn.fit(X_train, y_train)
            y_pred_val = knn.predict(X_val)
            metrics = calculate_metrics(y_val, y_pred_val)
            
            knn_results.append({
                'k': k,
                'metric': metric,
                'weighted': weighted,
                'accuracy': metrics['accuracy'],
                'f1_score': metrics['f1_score']
            })
            
            print(f"k={k:2d}, metric={metric:10s}, weighted={str(weighted):5s}: "
                  f"Acc={metrics['accuracy']:.4f}, F1={metrics['f1_score']:.4f}")

knn_results_df = pd.DataFrame(knn_results)
best_knn_idx = knn_results_df['f1_score'].idxmax()
best_knn_params = knn_results_df.iloc[best_knn_idx]

print(f"\nBest k-NN configuration:")
print(f"k={int(best_knn_params['k'])}, metric={best_knn_params['metric']}, "
      f"weighted={best_knn_params['weighted']}")
print(f"Validation F1-Score: {best_knn_params['f1_score']:.4f}")

### Train Best k-NN Model and Evaluate

In [ ]:
# Train best k-NN model
knn_best = KNNClassifier(
    k=int(best_knn_params['k']),
    distance_metric=best_knn_params['metric'],
    weighted=best_knn_params['weighted']
)
knn_best.fit(X_train, y_train)

# Evaluate on all sets
y_pred_train_knn = knn_best.predict(X_train)
y_pred_val_knn = knn_best.predict(X_val)
y_pred_test_knn = knn_best.predict(X_test)
y_proba_test_knn = knn_best.predict_proba(X_test)

metrics_train_knn = calculate_metrics(y_train, y_pred_train_knn)
metrics_val_knn = calculate_metrics(y_val, y_pred_val_knn)
metrics_test_knn = calculate_metrics(y_test, y_pred_test_knn, y_proba_test_knn)

print_metrics(metrics_test_knn, "k-NN (Test Set)")

# Learning curve for k-NN
knn_train_scores = []
knn_val_scores = []

for k in k_values:
    knn_temp = KNNClassifier(k=k, distance_metric=best_knn_params['metric'],
                            weighted=best_knn_params['weighted'])
    knn_temp.fit(X_train, y_train)
    train_pred = knn_temp.predict(X_train)
    val_pred = knn_temp.predict(X_val)
    knn_train_scores.append(calculate_metrics(y_train, train_pred)['accuracy'])
    knn_val_scores.append(calculate_metrics(y_val, val_pred)['accuracy'])

plt.figure(figsize=(10, 6))
plot_learning_curve(knn_train_scores, knn_val_scores, k_values, 'k',
                   'k-NN Learning Curve')
plt.savefig('knn_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Random Forest Classifier

### Hyperparameter Tuning

We'll tune Random Forest with different max_depth values and split criteria (Gini, Entropy).

In [ ]:
# Tune Random Forest
print("="*60)
print("Tuning Random Forest Classifier")
print("="*60)

max_depths = [3, 5, 7, 10, None]
rf_results = []

for criterion in ['gini', 'entropy']:
    for max_depth in max_depths:
        rf = RandomForest(n_trees=5, max_depth=max_depth, criterion=criterion, random_state=42)
        rf.fit(X_train, y_train)
        y_pred_val = rf.predict(X_val)
        metrics = calculate_metrics(y_val, y_pred_val)
        
        rf_results.append({
            'max_depth': max_depth if max_depth else 'None',
            'criterion': criterion,
            'accuracy': metrics['accuracy'],
            'f1_score': metrics['f1_score']
        })
        
        depth_str = str(max_depth) if max_depth else 'None'
        print(f"max_depth={depth_str:5s}, criterion={criterion:8s}: "
              f"Acc={metrics['accuracy']:.4f}, F1={metrics['f1_score']:.4f}")

rf_results_df = pd.DataFrame(rf_results)
best_rf_idx = rf_results_df['f1_score'].idxmax()
best_rf_params = rf_results_df.iloc[best_rf_idx]

print(f"\nBest Random Forest configuration:")
print(f"max_depth={best_rf_params['max_depth']}, criterion={best_rf_params['criterion']}")
print(f"Validation F1-Score: {best_rf_params['f1_score']:.4f}")

### Train Best Random Forest Model and Evaluate

In [ ]:
# Train best Random Forest model
max_depth_best = None if best_rf_params['max_depth'] == 'None' else int(best_rf_params['max_depth'])
rf_best = RandomForest(
    n_trees=5,
    max_depth=max_depth_best,
    criterion=best_rf_params['criterion'],
    random_state=42
)
rf_best.fit(X_train, y_train)

# Evaluate on all sets
y_pred_train_rf = rf_best.predict(X_train)
y_pred_val_rf = rf_best.predict(X_val)
y_pred_test_rf = rf_best.predict(X_test)
y_proba_test_rf = rf_best.predict_proba(X_test)

metrics_train_rf = calculate_metrics(y_train, y_pred_train_rf)
metrics_val_rf = calculate_metrics(y_val, y_pred_val_rf)
metrics_test_rf = calculate_metrics(y_test, y_pred_test_rf, y_proba_test_rf)

print_metrics(metrics_test_rf, "Random Forest (Test Set)")

# Feature importance
rf_importance = rf_best.get_feature_importance(feature_names)
plt.figure(figsize=(10, 6))
plot_feature_importance(rf_importance, 'Random Forest Feature Importance')
plt.savefig('rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# Learning curve for Random Forest
rf_train_scores = []
rf_val_scores = []

for md in [3, 5, 7, 10, 15]:
    rf_temp = RandomForest(n_trees=5, max_depth=md,
                          criterion=best_rf_params['criterion'],
                          random_state=42)
    rf_temp.fit(X_train, y_train)
    train_pred = rf_temp.predict(X_train)
    val_pred = rf_temp.predict(X_val)
    rf_train_scores.append(calculate_metrics(y_train, train_pred)['accuracy'])
    rf_val_scores.append(calculate_metrics(y_val, val_pred)['accuracy'])

plt.figure(figsize=(10, 6))
plot_learning_curve(rf_train_scores, rf_val_scores, [3, 5, 7, 10, 15], 'Max Depth',
                   'Random Forest Learning Curve')
plt.savefig('rf_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Naive Bayes Classifier

### Train and Evaluate Naive Bayes

In [ ]:
# Train Naive Bayes
print("="*60)
print("Training Naive Bayes Classifier")
print("="*60)

nb = NaiveBayes()
nb.fit(X_train, y_train)

# Evaluate on all sets
y_pred_train_nb = nb.predict(X_train)
y_pred_val_nb = nb.predict(X_val)
y_pred_test_nb = nb.predict(X_test)
y_proba_test_nb = nb.predict_proba(X_test)

metrics_train_nb = calculate_metrics(y_train, y_pred_train_nb)
metrics_val_nb = calculate_metrics(y_val, y_pred_val_nb)
metrics_test_nb = calculate_metrics(y_test, y_pred_test_nb, y_proba_test_nb)

print_metrics(metrics_test_nb, "Naive Bayes (Test Set)")

## 4. Model Comparison and Visualization

### Confusion Matrices

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_confusion_matrix(y_test, y_pred_test_knn, title='k-NN Confusion Matrix', ax=axes[0])
plot_confusion_matrix(y_test, y_pred_test_rf, title='Random Forest Confusion Matrix', ax=axes[1])
plot_confusion_matrix(y_test, y_pred_test_nb, title='Naive Bayes Confusion Matrix', ax=axes[2])

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

### ROC Curves

In [ ]:
# ROC curves for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_roc_curve(y_test, y_proba_test_knn, title='k-NN ROC Curve', ax=axes[0])
plot_roc_curve(y_test, y_proba_test_rf, title='Random Forest ROC Curve', ax=axes[1])
plot_roc_curve(y_test, y_proba_test_nb, title='Naive Bayes ROC Curve', ax=axes[2])

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

### Performance Comparison

In [ ]:
# Comparison of all models
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
model_names = ['k-NN', 'Random Forest', 'Naive Bayes']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    test_scores = [
        metrics_test_knn[metric],
        metrics_test_rf[metric],
        metrics_test_nb[metric]
    ]
    
    bars = ax.bar(model_names, test_scores, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'{metric.replace("_", " ").title()} Comparison (Test Set)')
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}',
               ha='center', va='bottom')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### Summary Table

In [ ]:
# Create summary table
summary_data = []
for name, metrics in [('k-NN', metrics_test_knn), 
                      ('Random Forest', metrics_test_rf), 
                      ('Naive Bayes', metrics_test_nb)]:
    summary_data.append({
        'Model': name,
        'Accuracy': f"{metrics['accuracy']:.4f}",
        'Precision': f"{metrics['precision']:.4f}",
        'Recall': f"{metrics['recall']:.4f}",
        'F1-Score': f"{metrics['f1_score']:.4f}",
        'ROC-AUC': f"{metrics.get('roc_auc', 'N/A')}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY (Test Set)")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

# Save results
summary_df.to_csv('results_summary.csv', index=False)
print("\nResults saved to 'results_summary.csv'")